# RNN/LSTM Image Captioning

Notebook ini menjalankan bagian Alvin: feature extraction Flickr8k, preprocessing caption, training decoder Keras RNN/LSTM pre-inject, scratch parity, inference, dan evaluasi BLEU-4/METEOR.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "ML-KPEZ" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(PROJECT_ROOT)
print(sys.executable)

/
/usr/local/bin/python3


In [2]:
import numpy as np
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.20.0
[]


## Paths and Toggles

In [3]:
IMAGES_DIR = PROJECT_ROOT / "data/raw/flickr8k/images"
CAPTIONS_PATH = PROJECT_ROOT / "data/raw/flickr8k/captions/captions.txt"
FEATURES_DIR = PROJECT_ROOT / "data/features/captioning"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/captioning"
MODELS_DIR = PROJECT_ROOT / "models/keras/captioning"
EXPERIMENTS_DIR = PROJECT_ROOT / "artifacts/experiments/captioning"
PREDICTIONS_DIR = PROJECT_ROOT / "artifacts/predictions/captioning"

RUN_FEATURE_EXTRACTION = False
RUN_PREPROCESSING = False
RUN_TRAINING = True
RUN_INIT_INJECT_TRAINING = False
RUN_EVALUATION = True
RUN_BATCH_INFERENCE = False

EVAL_LIMIT_IMAGES = None
EVAL_BACKENDS = "keras,scratch"
MAX_CAPTION_LENGTHS = "10,20,38"
BATCH_INFERENCE_IMAGE_IDS = []

print("images", IMAGES_DIR.exists())
print("captions", CAPTIONS_PATH.exists())
print("features", (FEATURES_DIR / "features.npy").exists())
print("processed", (PROCESSED_DIR / "train.npz").exists())


images False
captions False
features False
processed False


## Feature Extraction

In [4]:
from tubes2_ml.captioning.feature_extraction import FeatureExtractionConfig, extract_flickr8k_features

RUN_FEATURE_EXTRACTION = True

if RUN_FEATURE_EXTRACTION:
    result = extract_flickr8k_features(
        FeatureExtractionConfig(
            images_dir=IMAGES_DIR,
            output_dir=FEATURES_DIR,
            encoder_name="inception_v3",
            batch_size=32,
            overwrite=True,
        )
    )
    print(result)
else:
    print("Feature extraction skipped.")

ModuleNotFoundError: No module named 'tubes2_ml'

## Caption Preprocessing

In [ ]:
from tubes2_ml.captioning.preprocessing import CaptionPreprocessingConfig, preprocess_captions

if RUN_PREPROCESSING:
    result = preprocess_captions(
        CaptionPreprocessingConfig(
            captions_path=CAPTIONS_PATH,
            output_dir=PROCESSED_DIR,
            train_size=6000,
            validation_size=1000,
            test_size=1000,
        )
    )
    print(result)
else:
    print("Preprocessing skipped.")

Preprocessing skipped.


## Training Grid: 12 RNN/LSTM Experiments

In [ ]:
from scripts.run_captioning_experiments import configs_from_yaml, generate_experiment_configs, run_grid

base_model_config, training_config, grid = configs_from_yaml(PROJECT_ROOT / "configs/captioning/hparam_grid.yaml")
experiment_configs = generate_experiment_configs(base_model_config, **grid)
print("Total experiments:", len(experiment_configs))
[config.name for config in experiment_configs]

Total experiments: 12


['rnn_layers1_hidden128',
 'rnn_layers1_hidden512',
 'rnn_layers2_hidden128',
 'rnn_layers2_hidden512',
 'rnn_layers3_hidden128',
 'rnn_layers3_hidden512',
 'lstm_layers1_hidden128',
 'lstm_layers1_hidden512',
 'lstm_layers2_hidden128',
 'lstm_layers2_hidden512',
 'lstm_layers3_hidden128',
 'lstm_layers3_hidden512']

In [ ]:
if RUN_TRAINING:
    training_results = run_grid(experiment_configs, training_config, skip_completed=True)
    print(json.dumps(training_results, indent=2, default=str))
else:
    print("Training skipped. Set RUN_TRAINING = True to train remaining captioning models.")

Training skipped. Set RUN_TRAINING = True to train remaining captioning models.


## Init-Inject Bonus Experiments


In [ ]:
init_base_config, init_training_config, init_grid = configs_from_yaml(PROJECT_ROOT / "configs/captioning/init_inject.yaml")
init_experiment_configs = generate_experiment_configs(init_base_config, **init_grid)
print("Total init-inject experiments:", len(init_experiment_configs))

if RUN_INIT_INJECT_TRAINING:
    init_training_results = run_grid(init_experiment_configs, init_training_config, skip_completed=True)
    print(json.dumps(init_training_results, indent=2, default=str))
else:
    print("Init-inject training skipped. Set RUN_INIT_INJECT_TRAINING = True to train remaining bonus models.")


## Scratch Forward Parity Check

In [ ]:
from tubes2_ml.captioning.models import CaptionDecoderConfig, build_preinject_decoder
from tubes2_ml.scratch.models.rnn_captioner import build_scratch_rnn_captioner_from_keras
from tubes2_ml.scratch.models.lstm_captioner import build_scratch_lstm_captioner_from_keras

for kind, builder in [("rnn", build_scratch_rnn_captioner_from_keras), ("lstm", build_scratch_lstm_captioner_from_keras)]:
    keras_model = build_preinject_decoder(
        CaptionDecoderConfig(
            vocab_size=13,
            feature_dim=7,
            max_caption_length=5,
            embed_dim=4,
            hidden_units=6,
            num_recurrent_layers=2,
            decoder_type=kind,
            name=f"test_{kind}",
        )
    )
    scratch_model = builder(keras_model)
    features = np.random.default_rng(42).normal(size=(3, 7)).astype("float32")
    tokens = np.array([[1, 4, 5, 0, 0], [1, 3, 2, 0, 0], [1, 8, 9, 10, 11]], dtype="int32")
    keras_out = keras_model.predict([features, tokens], verbose=0)
    scratch_out = scratch_model.forward(features, tokens)
    print(kind, keras_out.shape, np.max(np.abs(keras_out - scratch_out)))

rnn (3, 5, 13) 2.2351742e-08
lstm (3, 5, 13) 1.4901161e-08


## Evaluation: BLEU-4, METEOR, Execution Time

In [ ]:
from scripts.evaluate_captioning_experiments import main as evaluate_captioning_main

if RUN_EVALUATION:
    import sys as _sys
    _old_argv = _sys.argv
    _sys.argv = [
        "evaluate_captioning_experiments.py",
        "--models-dir", str(MODELS_DIR),
        "--processed-dir", str(PROCESSED_DIR),
        "--features-dir", str(FEATURES_DIR),
        "--captions-path", str(CAPTIONS_PATH),
        "--split", "test",
        "--limit-images", str(EVAL_LIMIT_IMAGES),
        "--backends", EVAL_BACKENDS,
        "--max-caption-lengths", MAX_CAPTION_LENGTHS,
        "--output-csv", str(EXPERIMENTS_DIR / "evaluation_results.csv"),
        "--output-json", str(EXPERIMENTS_DIR / "evaluation_results.json"),
        "--summary-json", str(EXPERIMENTS_DIR / "evaluation_summary.json"),
        "--qualitative-json", str(PREDICTIONS_DIR / "qualitative_samples.json"),
        "--predictions-dir", str(PREDICTIONS_DIR),
    ]
    try:
        evaluate_captioning_main()
    finally:
        _sys.argv = _old_argv
else:
    print("Evaluation skipped. Set RUN_EVALUATION = True after models are trained.")

Evaluation skipped. Set RUN_EVALUATION = True after models are trained.


## Results Summary

In [ ]:
import csv

summary_path = EXPERIMENTS_DIR / "evaluation_summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))

results_path = EXPERIMENTS_DIR / "evaluation_results.csv"
if results_path.exists():
    with results_path.open("r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    rows_sorted = sorted(rows, key=lambda row: float(row["bleu4"]), reverse=True)
    for row in rows_sorted[:10]:
        print(row)
else:
    print("No evaluation results yet.")

No evaluation results yet.


## Qualitative Samples

In [ ]:
qualitative_path = PREDICTIONS_DIR / "qualitative_samples.json"
if qualitative_path.exists():
    payload = json.loads(qualitative_path.read_text(encoding="utf-8"))
    samples = payload.get("samples", [])
    print("source:", payload.get("source_result", {}))
elif (prediction_files := sorted(PREDICTIONS_DIR.glob("*.json"))):
    samples = json.loads(prediction_files[0].read_text(encoding="utf-8"))[:10]
    print(prediction_files[0])
else:
    samples = []
    print("No prediction files yet.")

for sample in samples[:10]:
    print("image_id:", sample["image_id"])
    print("prediction:", sample["caption"])
    print("references:", sample.get("references", [])[:2])
    print()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 8)

## Batch Caption Inference


In [ ]:
from tubes2_ml.captioning.inference import generate_captions

if RUN_BATCH_INFERENCE:
    candidate_models = sorted(MODELS_DIR.glob("*.keras"))
    if not candidate_models:
        raise FileNotFoundError(f"No trained models found in {MODELS_DIR}")
    image_ids = BATCH_INFERENCE_IMAGE_IDS
    if not image_ids:
        split = np.load(PROCESSED_DIR / "test.npz")
        image_ids = list(dict.fromkeys(str(image_id) for image_id in split["image_ids"]))[:5]
    batch_predictions = generate_captions(
        model_path=candidate_models[0],
        vocabulary_path=PROCESSED_DIR / "vocabulary.json",
        features_dir=FEATURES_DIR,
        image_ids=image_ids,
        max_caption_length=38,
        backend="scratch",
        search="greedy",
    )
    print(json.dumps(batch_predictions, indent=2))
else:
    print("Batch inference skipped. Set RUN_BATCH_INFERENCE = True after at least one pre-inject model is trained.")
